# Week 3 Assignment

# Mini Project : Customer Sales Insights

## Objective

The objective of this mini project is to analyse customer sales data using SQL.

Different SQL concepts such as Subqueries, CTEs, Window Functions and JOIN operations are applied to answer important business questions.

## Step 1 : Import Required Libraries

The required Python libraries are imported to execute SQL queries and display the results.

- pandas is used for displaying query results.
- sqlite3 is used for connecting to the SQLite database.

In [1]:
# Import pandas library
import pandas as pd

# Import sqlite3 library
import sqlite3

## Step 2 : Connect to the Database

Connect to the existing SQLite database created during the data setup process.

In [2]:
# Connect to SQLite database

conn = sqlite3.connect("../superstore.db")

cursor = conn.cursor()

## Step 3 : Verify Available Tables

Verify that the required tables are present before executing the final query.

In [3]:
pd.read_sql("""

SELECT name

FROM sqlite_master

WHERE type='table';

""", conn)

,name
0,customers
1,products
2,superstore_raw
3,orders


### Observation

The required tables are available in the SQLite database.

This confirms that the database setup has been completed successfully and the data is ready for business analysis.

## Project Introduction

In this mini project, customer sales data is analysed to answer important business questions.

Different SQL concepts such as JOIN, Aggregate Functions, Subqueries, Common Table Expressions (CTEs) and Window Functions are combined to generate meaningful business insights.

The analysis helps identify valuable customers, customer purchasing behaviour and revenue contribution.

## Business Question 1

### Problem Statement 

Who are the top 5 customers based on total sales?

### Understanding the Problem

A customer may place multiple orders.

The sales value of all orders placed by the same customer must be added together.

After calculating the total sales for every customer, the customers are arranged in descending order.

Finally, the top five customers with the highest sales are displayed.

### Approach

1. Calculate total sales for every customer using the SUM() function.
2. Group the records using customer name.
3. Sort the results in descending order of total sales.
4. Display only the first five customers.

In [4]:
# Business Question 1
# Display top 5 customers based in total sales

query1 = """ 
SELECT 
c.customer_name, 
SUM(o.sales) AS total_sales 
FROM orders o 
JOIN customers c 
ON o.customer_id = c.customer_id 
GROUP BY c.customer_name
ORDER BY total_sales DESC 
LIMIT 5; 
""" 
pd.read_sql(query1,conn)


,customer_name,total_sales
0,Ken Lonsdale,1247420.152
1,Sanjit Engle,1074430.544
2,Clay Ludtke,1044532.416
3,Adrian Barton,1042097.112
4,Sanjit Chand,1018248.048


### Observation

The query displays the top five customers having the highest total sales.

These customers contribute significantly to the company's revenue and represent the most valuable customer segment.

## Business Question 2

### Problem Statement 

Who are the bottom 5 customers based on total sales?

### Understanding the Problem

Not every customer contributes equally to the business revenue.

Some customers have placed very few orders or purchased products with low sales value.

This analysis identifies the five customers having the lowest total sales.

### Approach

- Join the customers table with the orders table.
- Calculate the total sales for each customer.
- Group the records using customer name.
- Arrange the customers in ascending order of total sales.
- Display the bottom five customers.

In [5]:
# Business Question 2
# Display bottom 5 customers based on total sales

query2 = """

SELECT
c.customer_name,
SUM(o.sales) AS total_sales
FROM orders o
JOIN customers c
ON o.customer_id = c.customer_id
GROUP BY c.customer_name
ORDER BY total_sales ASC
LIMIT 5;
"""
pd.read_sql(query2, conn)


,customer_name,total_sales
0,Lela Donovan,42.432
1,Thais Sissman,77.328
2,Carl Jackson,132.160
3,Mitch Gastineau,133.912
4,Roy Skaria,357.248


### Observation

The query displays the five customers with the lowest total sales.

These customers contribute the least towards the overall business revenue.

## Business Question 3

### Problem Statement 

Which customers made only one order?

### Understanding the Problem

Some customers purchase only once and never return.

Identifying such customers helps the business understand one-time buyers and improve customer retention.

### Approach

1. Join the customers table with the orders table.
2. Count the total number of orders placed by each customer.
3. Group the records by customer name.
4. Filter customers having exactly one order using the HAVING clause.


In [6]:
# Business Question 3
# Display customers who placed only one order

query3 = """

SELECT
c.customer_id,
c.customer_name,
COUNT(DISTINCT o.order_id) AS total_orders
FROM orders o
JOIN customers c
ON o.customer_id = c.customer_id
GROUP BY
c.customer_id,
c.customer_name
HAVING COUNT(DISTINCT o.order_id) = 1;
"""

pd.read_sql(query3, conn)

,customer_id,customer_name,total_orders
0,AO-10810,Anthony O'Donnell,1
1,AR-10570,Anemone Ratner,1
2,CJ-11875,Carl Jackson,1
3,JC-15385,Jenna Caffey,1
4,JR-15700,Jocasta Rupert,1
5,LD-16855,Lela Donovan,1
6,MG-18205,Mitch Gastineau,1
7,PH-18790,Patricia Hirasaki,1
8,RE-19405,Ricardo Emerson,1
9,RM-19750,Roland Murray,1


### Observation

The query identifies customers who have placed exactly one unique order.

The COUNT(DISTINCT order_id) function ensures that each order is counted only once, even if it contains multiple products.

The resulting customers represent one-time buyers in the dataset.

## Business Question 4

### Problem Statement 

Which customers have above-average total sales?

### Understanding the Problem

Every customer contributes differently to the overall sales.

First, the total sales of each customer are calculated.

Then, the average of these total sales values is determined.

Finally, only those customers whose total sales are greater than the average are displayed.

### Approach

1. Calculate total sales for every customer using a Common Table Expression (CTE).
2. Join the calculated result with the customers table.
3. Calculate the average total sales.
4. Filter customers whose total sales are greater than the average.

In [7]:
# Business Question 4
# Display customers whose total sales are above average

query4 = """

WITH customer_sales AS
(
SELECT
customer_id,
SUM(sales) AS total_sales
FROM orders
GROUP BY customer_id
)
SELECT 
c.customer_name,
cs.total_sales
FROM customer_sales cs
JOIN customers c
ON cs.customer_id = c.customer_id
WHERE cs.total_sales >
(
SELECT AVG(total_sales)
FROM customer_sales
)
ORDER BY cs.total_sales DESC;
"""
pd.read_sql(query4, conn)


,customer_name,total_sales
0,Sean Miller,25043.050
1,Sean Miller,25043.050
2,Sean Miller,25043.050
3,Sean Miller,25043.050
4,Sean Miller,25043.050
...,...,...
17811,Craig Yedwab,2900.026
17812,Craig Yedwab,2900.026
17813,Craig Yedwab,2900.026
17814,Craig Yedwab,2900.026


### Observation

The query displays customers whose total sales are greater than the average customer sales.

These customers contribute significantly to the company's overall revenue.

## Business Question 5

### Problem Statement 

What is the highest order value for each customer?

### Understanding the Problem

A customer may place multiple orders with different sales values.

The objective is to identify the highest sales value among all orders placed by each customer.

This helps in understanding the maximum purchase value of every customer.

### Approach

1. Join the customers and orders tables.
2. Group the records by customer.
3. Use the MAX() function to identify the highest sales value.
4. Display the customer name and highest order value.

In [8]:
# Business Question 5
# Display highest order value for each customer

query5 = """

SELECT
c.customer_name,
MAX(o.sales) AS highest_order_value
FROM orders o
JOIN customers c
ON o.customer_id = c.customer_id
GROUP BY 
c.customer_id,
c.customer_name
ORDER BY highest_order_value DESC;

"""

pd.read_sql(query5, conn)

,customer_name,highest_order_value
0,Sean Miller,22638.480
1,Tamara Chand,17499.950
2,Raymond Buch,13999.960
3,Tom Ashbrook,11199.968
4,Hunter Lopez,10499.970
...,...,...
788,Carl Jackson,16.520
789,Mitch Gastineau,12.320
790,Roy Skaria,9.648
791,Lela Donovan,5.304


### Observation 

The query displays the highest order value placed by each customer.

It highlights the maximum purchase amount recorded for every customer in the dataset.

## Key Learning

While working on this mini project, I learned how different SQL concepts can be combined to solve practical business problems.

Using JOIN, Aggregate Functions, Subqueries, CTEs and Window Functions improved my understanding of SQL-based data analysis and business reporting.

## Conclusion

This mini project successfully analysed customer sales data using SQL techniques.

The analysis identified top-performing customers, low-performing customers, one-time buyers, customers with above-average sales and the highest order value for each customer.

The results demonstrate how SQL can be used to transform raw business data into meaningful insights that support customer relationship management and business decision-making.

## Reflection

Working on this mini project helped me understand how SQL can be applied to solve real-world business problems.

Combining multiple SQL concepts improved my analytical thinking and strengthened my ability to extract meaningful insights from business data.

This project also increased my confidence in writing structured and efficient SQL queries.

In [9]:
# Close the database connection

conn.close()